# 00 - Setup: `bi_course` schema and 3-table sample dataset

**One-time setup notebook for the BI on the Lakehouse crash course (UBB Cluj).**

Run every cell top-to-bottom. After this notebook completes you will have:

- `workspace.bi_course` schema (Unity Catalog).
- `dim_product` (6 rows), `dim_region` (4 rows), `fact_sales` (12 rows) - all Delta.

**SSIS parallel.** What you are doing here is the equivalent of running
`CREATE DATABASE bi_course; CREATE TABLE …; INSERT …;` in SSMS, except the
namespace lives in **Unity Catalog** (think SSISDB) and the storage engine
is **Delta Lake** (Parquet + transaction log).

**Free Edition prerequisites.**

- Notebook attached to *Serverless* compute (top-right of the notebook bar).
- A serverless SQL warehouse running (only needed if you also use SQL Editor).


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bi_course
    COMMENT 'BI on the Lakehouse - 2-hour crash course (UBB Cluj)';

USE workspace.bi_course;


## `dim_product`


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.bi_course.dim_product (
    product_id   INT      NOT NULL,
    product_name STRING   NOT NULL,
    category     STRING   NOT NULL,
    list_price   DECIMAL(10, 2) NOT NULL
) USING DELTA;

INSERT INTO workspace.bi_course.dim_product VALUES
    (1, 'Laptop 14"',     'Computers',   1200.00),
    (2, 'Laptop 16"',     'Computers',   1800.00),
    (3, 'Wireless Mouse', 'Accessories',   25.00),
    (4, 'Mech. Keyboard', 'Accessories',   80.00),
    (5, 'Monitor 27"',    'Displays',     300.00),
    (6, 'Monitor 32"',    'Displays',     520.00);


## `dim_region`


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.bi_course.dim_region (
    region_id   INT    NOT NULL,
    region_name STRING NOT NULL,
    country     STRING NOT NULL
) USING DELTA;

INSERT INTO workspace.bi_course.dim_region VALUES
    (1, 'North', 'Romania'),
    (2, 'South', 'Romania'),
    (3, 'East',  'Romania'),
    (4, 'West',  'Romania');


## `fact_sales`


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.bi_course.fact_sales (
    order_id    BIGINT  NOT NULL,
    order_date  DATE    NOT NULL,
    product_id  INT     NOT NULL,
    region_id   INT     NOT NULL,
    quantity    INT     NOT NULL,
    revenue     DECIMAL(12, 2) NOT NULL
) USING DELTA;

INSERT INTO workspace.bi_course.fact_sales VALUES
    (1001, DATE'2026-01-15', 1, 1, 1, 1200.00),
    (1002, DATE'2026-01-15', 3, 1, 2,   50.00),
    (1003, DATE'2026-01-16', 1, 2, 1, 1100.00),
    (1004, DATE'2026-01-16', 4, 2, 1,   80.00),
    (1005, DATE'2026-01-17', 5, 4, 1,  300.00),
    (1006, DATE'2026-01-17', 3, 4, 1,   25.00),
    (1007, DATE'2026-01-18', 2, 3, 1, 1800.00),
    (1008, DATE'2026-01-18', 6, 3, 1,  520.00),
    (1009, DATE'2026-01-19', 1, 1, 2, 2400.00),
    (1010, DATE'2026-01-19', 5, 2, 1,  300.00),
    (1011, DATE'2026-01-20', 4, 4, 3,  240.00),
    (1012, DATE'2026-01-20', 2, 3, 1, 1750.00);


## Smoke test

Both queries should run in well under 5 seconds on a warm warehouse.


In [ ]:
%sql
SELECT COUNT(*) AS rows_in_fact FROM workspace.bi_course.fact_sales;  -- expect 12


In [ ]:
%sql
SELECT
    f.order_id,
    p.product_name,
    r.region_name,
    f.revenue
FROM workspace.bi_course.fact_sales       f
JOIN workspace.bi_course.dim_product      p USING (product_id)
JOIN workspace.bi_course.dim_region       r USING (region_id)
ORDER BY f.order_id
LIMIT 5;


Five neatly joined rows? You are ready for **Module 1**:
open `01_lakehouse_basics.ipynb`.
